# Module 6: Resilient Pipelines

**Workshop Track:** 300-Level  
**Prerequisites:** Modules 1 through 5 complete

---

Module 5 taught us how to submit work asynchronously and retrieve it later. That's a powerful pattern -- until the network hiccups, the API rate limit kicks in, or a server blips at exactly the wrong moment. In production, "works in my notebook" and "works at 3 AM when no one is watching" are two very different things.

Module 6 is about closing that gap. We are going to cover the the SDK error model — a typed exception hierarchy from `fundamental.exceptions` where each exception class tells you exactly how to respond. We will build retry logic with exponential backoff that handles transient failures without hammering the API, manage long-running operations with deadline-based polling instead of client-side timeouts, and assemble all of it into a resilient polling loop that can survive the kinds of failures that only show up in production.

The goal is not to make your code paranoid. It is to make it predictable -- so that when something goes wrong, your pipeline recovers gracefully instead of leaving half-completed work scattered across the system.

## Learning Objectives

By the end of this notebook you will:

- Understand the typed exception hierarchy and which exceptions are retryable
- Know which errors are retryable vs. fatal based on the exception class
- Implement exponential backoff with jitter for transient error recovery
- Manage long-running operations with deadline-based polling instead of client-side timeouts
- Build a resilient async polling loop that handles failures without losing in-flight work
- Combine all three patterns into a production-grade pipeline wrapper

---

## Setup

We will use the same enriched feature set from Modules 3 through 5. Your `SLIM_MODEL_ID` and `TOP_FEATURES` from earlier modules are retrieved automatically from the workshop state file on your Google Drive — no manual copy-paste needed.

If the retrieval fails (e.g., you skipped an earlier module), run the earlier modules first (approving the Drive mount), or paste the printed value when prompted. You can also look up your models manually with `client.models.list()`.

In [ ]:
# ============================================================================
# Workshop bootstrap — run this first. Safe to re-run. Identical in every module.
#
# In Google Colab, add two secrets via the key icon in the left sidebar
# (toggle "Notebook access" on for each):
#   • FUNDAMENTAL_API_KEY            — your NEXUS API key (ak_...)
#   • CLOUDSMITH_FUNDAMENTAL_TOKEN   — token to install the Fundamental SDK
#
# Each Colab notebook runs in its own fresh VM, so the modules share state
# (model IDs, feature lists) through a small JSON file on your Google Drive.
# This cell asks to mount Drive — approve the popup. If you decline, later
# modules will ask you to paste the values printed by earlier ones instead.
# See README.md for details.
# ============================================================================
import os, sys, json, subprocess
from pathlib import Path

# TODO at go-live: switch REPO to "Fundamental-Technologies/introduction-to-nexus"
REPO = "jawhnycooke/nexus_onboarding_workshop"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Dataset -----------------------------------------------------------------
# In Colab we sparse-fetch ONLY the dataset folder; the notebook itself is
# loaded straight from GitHub by Colab, so nothing else is needed on disk.
if IN_COLAB:
    _data_repo = Path("/content/workshop_data")
    DATASET_DIR = _data_repo / "dataset"
    if not DATASET_DIR.is_dir():
        print("Fetching the workshop dataset…")
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--filter=blob:none",
                        "--sparse", f"https://github.com/{REPO}.git", str(_data_repo)],
                       check=True)
        subprocess.run(["git", "-C", str(_data_repo), "sparse-checkout", "set", "dataset"],
                       check=True)
    WORKSHOP_ROOT = Path("/content")
else:
    # Locally the notebooks live in workshop_colab/ inside the cloned repo,
    # and the dataset sits at the repo root.
    _here = Path.cwd().resolve()
    _repo_root = next((p for p in [_here, *_here.parents]
                       if (p / "workshop_colab").is_dir() and (p / "dataset").is_dir()), None)
    if _repo_root is None:
        raise RuntimeError(
            "Could not locate the repository root. Launch Jupyter from inside the "
            "cloned repository so the dataset/ folder can be found."
        )
    DATASET_DIR = _repo_root / "dataset"
    WORKSHOP_ROOT = _repo_root / "workshop_colab"

os.chdir(WORKSHOP_ROOT)

# --- Cross-module state ------------------------------------------------------
# Model IDs and the selected feature list pass between modules through one JSON
# file. In Colab it lives on your Google Drive so it survives across notebooks.
_DRIVE_MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        _state_dir = Path("/content/drive/MyDrive/nexus_workshop")
        _state_dir.mkdir(parents=True, exist_ok=True)
        STATE_FILE = _state_dir / "_workshop_state.json"
        _DRIVE_MOUNTED = True
    except Exception as _e:
        print(f"Google Drive not mounted ({_e}).")
        print("State will NOT persist between notebooks — later modules will ask "
              "you to paste the values printed by earlier ones.")
        STATE_FILE = Path("/content/_workshop_state.json")
else:
    STATE_FILE = WORKSHOP_ROOT / "_workshop_state.json"


def save_state(key, value):
    state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    state[key] = value
    STATE_FILE.write_text(json.dumps(state, indent=2))
    print(f"Saved '{key}' = {value!r}")
    if IN_COLAB and not _DRIVE_MOUNTED:
        print("  (Drive is not mounted — copy this value; the next module will ask for it.)")


def load_state(key, default=None):
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text()).get(key, default)
    return default


def require_state(key, produced_by):
    val = load_state(key)
    if val is None:
        print(f"'{key}' was not found in the workshop state file ({STATE_FILE}).")
        print(f"It is produced by {produced_by}. If you ran that module without "
              "mounting Drive, paste the value it printed.")
        val = input(f"Paste {key} (or press Enter to abort): ").strip().strip("'\"")
        if not val:
            raise RuntimeError(
                f"'{key}' unavailable. Run {produced_by} first (approving the Drive "
                "mount), or re-run this cell and paste the printed value. See README.md."
            )
        save_state(key, val)
    return val


def _get_secret(name, required=True):
    val = os.getenv(name)
    if not val and IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    if required and not val:
        raise RuntimeError(
            f"Missing secret '{name}'.\n"
            "  • In Colab: open the key icon (Secrets) in the left sidebar, add a "
            f"secret named '{name}', and turn on 'Notebook access'.\n"
            f"  • Locally: export {name} in your shell before launching Jupyter.\n"
            "See README.md for details."
        )
    return val


# --- SDK install (Colab only; locally the SDK is installed during setup).
# Colab already ships pandas, numpy, scikit-learn, matplotlib and xgboost,
# so the Fundamental SDK is the only package to install.
try:
    import fundamental  # noqa: F401
except ImportError:
    if not IN_COLAB:
        raise RuntimeError(
            "Fundamental SDK not found. Install it locally (see README.md) before running."
        )
    _token = _get_secret("CLOUDSMITH_FUNDAMENTAL_TOKEN")
    print("Installing the Fundamental SDK…")
    _index = f"https://dl.cloudsmith.io/{_token}/fundamental/fundamental-client/python/simple/"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fundamental-client==0.10.0", "--extra-index-url", _index], check=True)

# --- Authentication ---
FUNDAMENTAL_API_KEY = _get_secret("FUNDAMENTAL_API_KEY")
os.environ["FUNDAMENTAL_API_KEY"] = FUNDAMENTAL_API_KEY

from fundamental import Fundamental, NEXUSClassifier, NEXUSRegressor, set_client
client = Fundamental()
set_client(client)

print(f"Workshop ready. Dataset: {DATASET_DIR}")
print(f"State file: {STATE_FILE}"
      + ("" if _DRIVE_MOUNTED or not IN_COLAB else "  (not persistent — Drive not mounted)"))
print(f"API key prefix: {FUNDAMENTAL_API_KEY[:8]}…")


In [2]:
import os
import time
import random
import json
import logging
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

from fundamental import Fundamental, NEXUSClassifier, set_client
from fundamental.exceptions import (
    NEXUSError,
    ValidationError,
    AuthenticationError,
    AuthorizationError,
    NotFoundError,
    RateLimitError,
    ServerError,
    NetworkError,
    RequestTimeoutError,
)

# Authentication and the shared client were configured by the bootstrap cell above.
print("Authentication: OK")

Authentication: OK


In [3]:
DATA_DIR = DATASET_DIR / "credit_risk"

train_raw   = pd.read_csv(f"{DATA_DIR}/borrowers_train.csv")
holdout_raw = pd.read_csv(f"{DATA_DIR}/borrowers_holdout.csv")
snapshots   = pd.read_csv(f"{DATA_DIR}/financial_snapshots.csv", parse_dates=["snapshot_date"])
assessments = pd.read_csv(f"{DATA_DIR}/credit_assessments.csv", parse_dates=["assessment_date"])
payments    = pd.read_csv(f"{DATA_DIR}/payment_events.csv", parse_dates=["payment_date"])

snapshots_latest = (
    snapshots.sort_values("snapshot_date", ascending=False)
    .drop_duplicates(subset="borrower_id", keep="first")
    [["borrower_id", "monthly_income_usd", "income_growth_pct",
      "collateral_score", "secondary_income_flag"]]
)
assessments_latest = (
    assessments.sort_values("assessment_date", ascending=False)
    .drop_duplicates(subset="borrower_id", keep="first")
    [["borrower_id", "creditworthiness_rating", "payment_behavior_score",
      "financial_stability_score", "lender_relationship_score",
      "credit_engagement_score", "debt_service_score"]]
)
payment_agg = (
    payments.groupby("borrower_id")
    .agg(
        total_payments=("payment_id", "count"),
        on_time_rate=("on_time", lambda x: (x == "Yes").mean()),
        avg_payment_usd=("amount_usd", "mean"),
    )
    .reset_index()
)

def enrich(df):
    out = df.copy()
    out = out.merge(snapshots_latest, on="borrower_id", how="left")
    out = out.merge(assessments_latest, on="borrower_id", how="left")
    out = out.merge(payment_agg, on="borrower_id", how="left")
    return out

train_enriched   = enrich(train_raw)
holdout_enriched = enrich(holdout_raw)

drop_cols    = ["borrower_id", "first_name", "last_name", "default_flag"]
all_features = [c for c in train_enriched.columns if c not in drop_cols]

# Apply the Module 3 feature prep: the date becomes the numeric account_tenure_days
# feature (NEXUS does not support datetime columns); categoricals pass to NEXUS as-is
def add_account_tenure(df, date_col="account_open_date", ref_date="2026-01-01"):
    """Convert the account-open date into a numeric tenure feature.

    NEXUS accepts numeric, boolean, string, and categorical columns — but not
    datetime columns. So instead of parsing the date, we derive an explicit
    numeric feature: the account's age in days at a fixed reference date.
    (A fixed reference keeps the feature stable across runs.)
    """
    out = df.copy()
    out["account_tenure_days"] = (
        pd.Timestamp(ref_date) - pd.to_datetime(out[date_col])
    ).dt.days
    return out.drop(columns=[date_col])


X_train_full = add_account_tenure(train_enriched[all_features])
X_holdout_full = add_account_tenure(holdout_enriched[all_features])

# Retrieve TOP_FEATURES from Module 4 via the shared workshop state file.
TOP_FEATURES = load_state(
    "TOP_FEATURES",
    default=['monthly_income_usd', 'avg_payment_usd', 'total_employment_years', 'age',
             'secondary_income_flag', 'account_tenure_days', 'marital_status', 'collateral_score',
             'occupation_sector', 'num_previous_lenders', 'distance_from_branch_miles',
             'credit_engagement_score', 'financial_stability_score', 'payment_behavior_score',
             'creditworthiness_rating'],
)
if not (isinstance(TOP_FEATURES, (list, tuple)) and set(TOP_FEATURES).issubset(X_train_full.columns)):
    raise RuntimeError(
        "TOP_FEATURES does not match the enriched feature columns. Run Module 4 in THIS "
        "Colab runtime first so it writes TOP_FEATURES to the shared workshop state file."
    )
print(f"Loaded {len(TOP_FEATURES)} features from Module 4")

X_train   = X_train_full[TOP_FEATURES]
X_holdout = X_holdout_full[TOP_FEATURES]
y_train   = train_enriched["default_flag"]
y_holdout = holdout_enriched["default_flag"]
print(f"Feature matrix ready: {X_train.shape[0]:,} rows x {X_train.shape[1]} features")

Loaded 15 features from Module 4
Feature matrix ready: 4,591 rows x 15 features


In [4]:
# Retrieve your model ID from Module 5 (written to the shared workshop state file).
# If the stored id is missing or stale, fall back to fitting a fresh baseline model.
SLIM_MODEL_ID = require_state("SLIM_MODEL_ID", "module_05")
try:
    nexus_baseline = NEXUSClassifier()
    nexus_baseline.load_model(SLIM_MODEL_ID)
    _ = nexus_baseline.predict_proba(X_holdout.head(1))
except Exception:
    print("Stored SLIM_MODEL_ID stale — fitting a fresh baseline model.")
    nexus_baseline = NEXUSClassifier()
    nexus_baseline.fit(X_train, y_train)
    SLIM_MODEL_ID = nexus_baseline.trained_model_id_

proba_baseline = nexus_baseline.predict_proba(X_holdout)
auc_baseline   = roc_auc_score(y_holdout, proba_baseline[:, 1])
print(f"Baseline model loaded. Holdout AUC: {auc_baseline:.4f}")

Baseline model loaded. Holdout AUC: 0.9441


---

## Part 1: The Error Model

### Two Layers of Defense

The Fundamental SDK organizes errors into two layers that tell you exactly where the failure happened and what to do about it:

**Layer 1: Client-side validation (`TypeError`, `ValueError`)**
These are raised immediately, before any network call leaves your machine. They mean something is wrong with the data you passed in — a DataFrame where an ndarray was expected, an empty dataset, mismatched sizes, or a missing API key. Fix your input and try again.

**Layer 2: Platform errors (`NEXUSError` and its subclasses)**
These are raised by `fit()`, `predict()`, `poll_*()`, and other API-bound calls when the platform refuses or fails the request. The exception class itself encodes the response category:

- **Retry** — `RateLimitError`, `NetworkError`, `RequestTimeoutError`, `ServerError`. Transient infrastructure issues. Retry with backoff.
- **Fix input** — `ValidationError`, `AuthenticationError`, `AuthorizationError`, `NotFoundError`. The problem is on your side. Do not retry. Fix your data, credentials, or model ID.
- **Escalate** — anything else inheriting from `NEXUSError`. Treat as a platform bug; capture `trace_id` and contact support.

You can `import` all of these from `fundamental.exceptions`. Catching on the type — `isinstance(exc, RateLimitError)` — is the recommended pattern.

### The Complete Exception Reference

Every platform error is one of these typed exceptions:

| Exception | HTTP | Category | What to do |
|-----------|------|----------|------------|
| `ValidationError` | 400 | fix input | Input data is malformed (wrong types, missing columns, etc.). Re-submitting the same data will fail again. |
| `AuthenticationError` | 401 | fix input | API key is invalid or missing. Check `FUNDAMENTAL_API_KEY`. |
| `AuthorizationError` | 403 | fix input | API key lacks permission for this operation. |
| `NotFoundError` | 404 | fix input | Model ID does not exist. Re-check `load_model` arguments. |
| `RateLimitError` | 429 | **retry** | API quota hit. Back off and retry. |
| `ServerError` | 500 | **retry** | Transient backend issue. Retry. |
| `NetworkError` | 503 | **retry** | Network connectivity glitch. Retry. |
| `RequestTimeoutError` | 504 | **retry** | Request timed out. Retry. |
| `NEXUSError` | — | base class | Catch-all. If something is in this layer that isn't more specific, treat as escalate-to-support. |

The practical rule: **catch on the type**. `isinstance(exc, (RateLimitError, NetworkError, RequestTimeoutError, ServerError))` tells you the call is safe to retry. All other `NEXUSError` subclasses mean: fix your input or escalate.

Every exception also carries `exc.trace_id` (a unique identifier the support team can use to look up your request) and `exc.details` (an optional dict with additional context). Always log `trace_id` when you catch one — it is the single most useful field for getting help fast.

In [5]:
# Demonstration: catch typed exceptions and inspect their attributes.
# Every NEXUSError carries .trace_id (str | None) and .details (dict, may be empty).

# Simulate a transient (retryable) error
try:
    raise RateLimitError("API quota exceeded for this minute")
except NEXUSError as exc:
    print(f"Type:           {type(exc).__name__}")
    print(f"Message:        {exc}")
    print(f"trace_id:       {getattr(exc, 'trace_id', None)}")
    print(f"details:        {getattr(exc, 'details', {})}")
    retryable = isinstance(exc, (RateLimitError, NetworkError, RequestTimeoutError, ServerError))
    print(f"Retryable?      {retryable}")

print()

# Simulate a user-error (do not retry)
try:
    raise NotFoundError("Model abc123 does not exist")
except NEXUSError as exc:
    print(f"Type:           {type(exc).__name__}")
    print(f"Message:        {exc}")
    print(f"trace_id:       {getattr(exc, 'trace_id', None)}")
    retryable = isinstance(exc, (RateLimitError, NetworkError, RequestTimeoutError, ServerError))
    print(f"Retryable?      {retryable}")

Type:           RateLimitError
Message:        HTTP 429: API quota exceeded for this minute
trace_id:       None
details:        {}
Retryable?      True

Type:           NotFoundError
Message:        HTTP 404: Model abc123 does not exist
trace_id:       None
Retryable?      False


**Catch on the type, not on a string code.** `isinstance(exc, RateLimitError)` is unambiguous and refactor-safe. Avoid scraping `str(exc)` for substrings — message text is not part of the SDK's stable interface.

Make it a habit to always log `exc.trace_id` whenever you catch a `NEXUSError`. That single field gives the support team the exact context they need to diagnose issues quickly.

---

## Part 2: Catching Exceptions in Practice

### The Basic Pattern

A well-structured exception handler for NEXUS operations follows a tiered catch in this order:

1. **Retryable typed exceptions** — `RateLimitError`, `NetworkError`, `RequestTimeoutError`, `ServerError`. Sleep with backoff and retry.
2. **`NEXUSError`** — anything else that came from the platform (`ValidationError`, `AuthenticationError`, etc.). Log `trace_id`, do not retry.
3. **`TypeError` / `ValueError`** — client-side validation failures; fix your data and re-raise.

Here is what that looks like for a synchronous `predict_proba` call:

In [6]:
# Set up a simple logger for this module
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("nexus_module06")

In [7]:
# Define the retryable exception types once — used by every retry helper below
RETRYABLE_EXCEPTIONS = (RateLimitError, NetworkError, RequestTimeoutError, ServerError)


def is_retryable(exc):
    """Check whether an exception represents a transient, retryable failure."""
    return isinstance(exc, RETRYABLE_EXCEPTIONS)


def safe_predict_proba(nexus, X, max_retries=3, base_delay=5.0):
    """
    Wrap predict_proba with basic retry logic for transient errors.

    Retries on RateLimitError, NetworkError, RequestTimeoutError, ServerError.
    All other NEXUSError subclasses (ValidationError, AuthenticationError,
    AuthorizationError, NotFoundError) are re-raised immediately — those need
    a fix to your input, not a retry.
    """
    attempt = 0

    while attempt <= max_retries:
        try:
            return nexus.predict_proba(X)

        except RETRYABLE_EXCEPTIONS as exc:
            attempt += 1
            trace_id = getattr(exc, "trace_id", None)
            if attempt > max_retries:
                log.error(f"All {max_retries} retries exhausted. trace_id={trace_id}")
                raise

            # Exponential backoff: 5s, 10s, 20s...
            wait = base_delay * (2 ** (attempt - 1))
            log.warning(
                f"Transient {type(exc).__name__} on attempt {attempt}/{max_retries}. "
                f"trace_id={trace_id}. Retrying in {wait:.0f}s."
            )
            time.sleep(wait)

        except NEXUSError as exc:
            # Non-retryable platform error (ValidationError, AuthenticationError, etc.)
            trace_id = getattr(exc, "trace_id", None)
            log.error(
                f"{type(exc).__name__}: {exc} (trace_id={trace_id}) — fix input or escalate, do not retry"
            )
            raise

        except (TypeError, ValueError) as exc:
            log.error(f"Client-side validation error: {exc} — fix your input data")
            raise


# Test the wrapper against our baseline model
proba = safe_predict_proba(nexus_baseline, X_holdout)
auc   = roc_auc_score(y_holdout, proba[:, 1])
print(f"Prediction succeeded. AUC: {auc:.4f}")

14:25:40 | INFO     | Uploading data, part 1/1 (33770 bytes)


Prediction succeeded. AUC: 0.9424


---

## Part 3: Retry Logic with Exponential Backoff

### Why Exponential Backoff?

Naive retry logic -- try again immediately -- makes things worse during a rate limit event. If you hit the limit and immediately retry five times, you are hammering the API just as it is trying to recover. That keeps the rate limit active longer and hurts other clients too.

Exponential backoff solves this by doubling the wait time with each failed attempt: 5 seconds, then 10, then 20, then 40. The total delay grows quickly, which gives the API breathing room. Adding a small random jitter (a fraction of a second) prevents multiple clients from all waking up at the same moment and thundering-herding the API again.

Think of it like a crowded coffee shop. If everyone rushes to the counter at once after a lull, you recreate the bottleneck. Jitter is the equivalent of people wandering up at slightly different times -- the line stays manageable.

In [8]:
def backoff_delay(attempt, base=5.0, cap=120.0, jitter=True):
    """
    Calculate exponential backoff delay.

    Parameters
    ----------
    attempt   : int   -- 1-based attempt number (1 = first retry)
    base      : float -- base delay in seconds (doubles each attempt)
    cap       : float -- maximum delay regardless of attempt count
    jitter    : bool  -- add up to 20% random variation to avoid thundering herd

    Returns
    -------
    float -- delay in seconds
    """
    delay = min(base * (2 ** (attempt - 1)), cap)
    if jitter:
        delay += random.uniform(0, delay * 0.2)
    return delay


# Visualize the backoff schedule
print("Backoff schedule (first 6 retries):")
print(f"{'Attempt':>8} | {'Delay (no jitter)':>18} | {'Approx with jitter':>20}")
print("-" * 54)
for i in range(1, 7):
    d_exact  = backoff_delay(i, jitter=False)
    d_jitter = backoff_delay(i, jitter=True)
    print(f"{i:>8} | {d_exact:>15.1f}s  | ~{d_jitter:>16.1f}s")

Backoff schedule (first 6 retries):
 Attempt |  Delay (no jitter) |   Approx with jitter
------------------------------------------------------
       1 |             5.0s  | ~             6.0s
       2 |            10.0s  | ~            10.6s
       3 |            20.0s  | ~            21.9s
       4 |            40.0s  | ~            44.7s
       5 |            80.0s  | ~            95.1s
       6 |           120.0s  | ~           129.1s


**New to decorators?** Start with the plain version. The loop below is the whole retry idea: try the call; if the failure is one of the retryable types, wait and try again; give up after a fixed number of attempts. The decorator that follows is the same logic packaged so any function can wear it — a decorator is a function that wraps another function.

In [9]:
def retry_predict_proba(estimator, X, max_retries=2, delay=5.0):
    """The plain retry loop. The decorator below generalizes exactly this."""
    for attempt in range(1, max_retries + 2):  # the first try plus max_retries retries
        try:
            return estimator.predict_proba(X)
        except RETRYABLE_EXCEPTIONS as exc:
            if attempt > max_retries:
                raise
            print(f"Attempt {attempt} hit {type(exc).__name__}; retrying in {delay:.0f}s")
            time.sleep(delay)

proba_plain = retry_predict_proba(nexus_baseline, X_holdout)
print(f"Probabilities back: {proba_plain.shape} (no retries needed this run)")

14:25:53 | INFO     | Uploading data, part 1/1 (33770 bytes)


Probabilities back: (1149, 2) (no retries needed this run)


### A Reusable Retry Decorator

Once you have `backoff_delay`, the natural next step is a general-purpose retry decorator so you do not have to wrap every call in boilerplate. The decorator below catches the four retryable exception types and retries with backoff. Everything else (including non-retryable `NEXUSError` subclasses) propagates immediately.

In [10]:
import functools


def with_retry(max_retries=3, base_delay=5.0):
    """
    Decorator factory that wraps any function with retry logic.

    Retries only on the retryable exception types
    (RateLimitError, NetworkError, RequestTimeoutError, ServerError).
    All other NEXUSError subclasses propagate immediately.

    Usage:
        @with_retry(max_retries=4, base_delay=10.0)
        def my_nexus_call(...):
            ...
    """
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_retries + 2):  # +2: initial attempt + retries
                try:
                    return fn(*args, **kwargs)
                except RETRYABLE_EXCEPTIONS as exc:
                    trace_id = getattr(exc, "trace_id", None)
                    if attempt > max_retries:
                        log.error(
                            f"{fn.__name__}: all {max_retries} retries exhausted. "
                            f"trace_id={trace_id}"
                        )
                        raise
                    wait = backoff_delay(attempt, base=base_delay)
                    log.warning(
                        f"{fn.__name__}: {type(exc).__name__} on attempt {attempt}/{max_retries}. "
                        f"trace_id={trace_id}. Retrying in {wait:.1f}s."
                    )
                    time.sleep(wait)
        return wrapper
    return decorator


# Apply the decorator to predict_proba
@with_retry(max_retries=3, base_delay=5.0)
def resilient_predict_proba(nexus, X):
    return nexus.predict_proba(X)


proba = resilient_predict_proba(nexus_baseline, X_holdout)
auc   = roc_auc_score(y_holdout, proba[:, 1])
print(f"Resilient prediction succeeded. AUC: {auc:.4f}")

14:26:05 | INFO     | Uploading data, part 1/1 (33770 bytes)


Resilient prediction succeeded. AUC: 0.9434


---

## Part 4: Managing Long-Running Operations

### Deadlines Instead of Client-Side Timeouts

The the SDK exposes timeout knobs on `Fundamental(timeout=..., fit_timeout=..., predict_timeout=..., feature_importance_timeout=...)`, but the server-side timeouts are the ones that actually decide when to fail a job. A slow or timed-out poll surfaces a `RequestTimeoutError` (retryable); the job keeps running server-side.

For your code, the right approach is **deadline-based polling**: set a wall-clock deadline for how long you are willing to wait, then poll in a loop until either the result arrives or the deadline passes. This is the same submit/poll pattern from Module 5, but now with a time budget.

This approach has two advantages over relying on a single client-side timeout:
1. **No wasted work.** If your poll loop times out, the job keeps running on Fundamental's infrastructure. You can resume polling later with the same task ID.
2. **Explicit control.** You decide exactly how long to wait based on your workload, not a library default.

In [11]:
# Deadline-based polling pattern for long-running operations
# This is the basic building block -- Part 5 adds full error handling

nexus_demo = NEXUSClassifier(mode="quality")

# Submit a training task
task_id = nexus_demo.submit_fit_task(X_train, y_train)
print(f"Task submitted: {task_id}")

deadline = time.time() + 3600  # 1 hour wall-clock deadline

while time.time() < deadline:
    try:
        result = nexus_demo.poll_fit_result(task_id)
        if result is not None:
            print(f"Training complete! Model ID: {nexus_demo.trained_model_id_}")
            break
    except RETRYABLE_EXCEPTIONS as exc:
        trace_id = getattr(exc, "trace_id", None)
        print(f"Transient {type(exc).__name__} during poll. trace_id={trace_id}. Waiting 30s...")
        time.sleep(30)
        continue
    time.sleep(20)
else:
    print(f"Deadline exceeded. Task ID preserved: {task_id}")

# Verify the model works
proba_demo = nexus_demo.predict_proba(X_holdout)
auc_demo   = roc_auc_score(y_holdout, proba_demo[:, 1])
print(f"Prediction under deadline polling succeeded. AUC: {auc_demo:.4f}")

14:26:18 | INFO     | Uploading data, part 1/1 (93912 bytes)


14:26:18 | INFO     | Uploading data, part 1/1 (1887 bytes)


Task submitted: c9e43ce9cf77281951acf3510ea64f0c


14:27:00 | INFO     | Uploading data, part 1/1 (33770 bytes)


Training complete! Model ID: 966ff1c1-0475-4140-96db-516e4640c5d6


Prediction under deadline polling succeeded. AUC: 0.9438


In [12]:
# The key insight: even if our poll loop exits (deadline, crash, notebook restart),
# the task_id is our recovery handle. We can resume polling in a new session:
#
#   nexus_resume = NEXUSClassifier()
#   result = nexus_resume.poll_fit_result(task_id)  # pick up where we left off
#
print(f"Task ID to save: {task_id}")
print("This ID remains valid regardless of what happens to your poll loop.")

Task ID to save: c9e43ce9cf77281951acf3510ea64f0c
This ID remains valid regardless of what happens to your poll loop.


### Task Lifecycle and What Errors Mean at Each Stage

Tasks move through a well-defined lifecycle: **CREATED** -> **PENDING** -> **IN_PROGRESS** -> **COMPLETE** | **FAILED**

A `RequestTimeoutError` during polling means the poll *connection* timed out, not that the job itself stopped. The job continues running on Fundamental's infrastructure. You can safely retry the poll with the same task ID.

A task that reaches `FAILED` with a transient failure can often be retried by re-submitting. But always check the exception type first — a `ValidationError` in the `FAILED` state means your data was bad, and re-submitting the same data will fail again.

---

## Part 5: A Resilient Async Poll Loop

### Putting It All Together

In Module 5 our poll loops were clean and optimistic — they assumed `poll_fit_result` would always return cleanly. In production, the poll itself can throw any of the transient exceptions. The network might drop for a moment, the API might briefly rate-limit you, or a connection might time out.

The resilient poll loop below handles all of this:

- Each poll call catches the retryable exception types and decides how to respond
- Non-retryable errors (e.g., `ValidationError`, `AuthenticationError`, `NotFoundError`) surface immediately without spinning forever
- The loop has a total wall-clock deadline so it cannot run indefinitely
- A consecutive error cap prevents infinite retry spirals on persistent failures
- **The task ID is preserved regardless of poll failures** — the job keeps running

**The poll loop as a state machine.** Before reading the function, hold this shape in mind:

    while the deadline has not passed:
        poll the task
          result ready          -> return it
          still running         -> reset the error counter, sleep, poll again
          retryable exception   -> count it; under the cap, sleep and poll again;
                                   at the cap, raise
    deadline passed             -> raise (the task id stays valid — resume later)

Two clocks run at once: wall-clock time against the deadline, and consecutive errors against the cap. A clean poll resets the error counter; only the deadline ends a healthy wait.

In [13]:
def resilient_poll_fit(
    nexus,
    task_id,
    poll_interval=20,
    max_poll_errors=5,
    deadline_seconds=3600,
):
    """
    Poll a fit task with full error handling.

    Parameters
    ----------
    nexus            : NEXUSClassifier  -- the estimator that submitted the task
    task_id          : str    -- task ID from submit_fit_task
    poll_interval    : int    -- seconds between poll attempts
    max_poll_errors  : int    -- consecutive transient errors before giving up
    deadline_seconds : int    -- total wall-clock timeout for the poll loop

    Returns
    -------
    The poll result (non-None value from poll_fit_result)

    Raises
    ------
    NEXUSError    -- on a non-retryable platform error
    RuntimeError  -- if the deadline is exceeded or too many consecutive transient errors
    """
    start_time       = time.time()
    consecutive_errs = 0
    poll_count       = 0

    log.info(f"Starting resilient poll for task {task_id}")

    while True:
        # Enforce the wall-clock deadline
        elapsed = time.time() - start_time
        if elapsed > deadline_seconds:
            raise RuntimeError(
                f"Poll deadline exceeded after {elapsed:.0f}s. "
                f"Task {task_id} is still running -- save the task ID and resume later."
            )

        poll_count += 1

        try:
            result = nexus.poll_fit_result(task_id)
            consecutive_errs = 0  # Reset error counter on a clean poll

            if result is not None:
                log.info(
                    f"Training complete after {elapsed:.0f}s and {poll_count} polls. "
                    f"Model ID: {nexus.trained_model_id_}"
                )
                return result

            log.info(f"Poll {poll_count}: still running ({elapsed:.0f}s elapsed). Next check in {poll_interval}s.")
            time.sleep(poll_interval)

        except RETRYABLE_EXCEPTIONS as exc:
            consecutive_errs += 1
            wait = backoff_delay(consecutive_errs, base=10.0)
            trace_id = getattr(exc, "trace_id", None)
            log.warning(
                f"{type(exc).__name__} on poll {poll_count} ({consecutive_errs} consecutive). "
                f"trace_id={trace_id}. Waiting {wait:.0f}s."
            )

            if consecutive_errs >= max_poll_errors:
                raise RuntimeError(
                    f"{max_poll_errors} consecutive poll errors. "
                    f"Last error: {type(exc).__name__}: {exc} (trace_id={trace_id}). "
                    f"Task {task_id} may still be running -- check again later."
                ) from exc

            time.sleep(wait)

        except NEXUSError as exc:
            # Non-retryable platform error (ValidationError, AuthenticationError, NotFoundError, ...)
            trace_id = getattr(exc, "trace_id", None)
            log.error(f"{type(exc).__name__} on poll: {exc} (trace_id={trace_id})")
            raise


print("resilient_poll_fit defined.")
print("Signature: resilient_poll_fit(nexus, task_id, poll_interval=20, max_poll_errors=5, deadline_seconds=3600)")

resilient_poll_fit defined.
Signature: resilient_poll_fit(nexus, task_id, poll_interval=20, max_poll_errors=5, deadline_seconds=3600)


Notice the message inside the `RuntimeError` when the deadline is exceeded: **save the task ID and resume later**. This is the most important resilience property the async model gives us. Even if your poll loop crashes or is killed, the training job keeps running. The task ID is your recovery handle.

If you hit a deadline or a run of poll errors, do not re-submit the training job. Just save the task ID, investigate the error, and resume polling in a new session.

In [14]:
# Run a full async training cycle using the resilient poll loop
nexus_resilient = NEXUSClassifier(mode="quality")

# Submit the fit task
task_id = nexus_resilient.submit_fit_task(X_train, y_train)
log.info(f"Task submitted: {task_id}")

# Poll with full error handling
resilient_poll_fit(
    nexus_resilient,
    task_id,
    poll_interval=20,
    max_poll_errors=5,
    deadline_seconds=900,  # 15-minute cap for workshop
)

# Evaluate
proba_resilient = nexus_resilient.predict_proba(X_holdout)
auc_resilient   = roc_auc_score(y_holdout, proba_resilient[:, 1])
print(f"\nResilient training complete.")
print(f"Model ID:    {nexus_resilient.trained_model_id_}")
print(f"Holdout AUC: {auc_resilient:.4f}")

14:27:12 | INFO     | Uploading data, part 1/1 (93912 bytes)


14:27:13 | INFO     | Uploading data, part 1/1 (1887 bytes)


14:27:13 | INFO     | Task submitted: 1ed9b7eb4b2ecd0d2727ca933d02798d


14:27:13 | INFO     | Starting resilient poll for task 1ed9b7eb4b2ecd0d2727ca933d02798d


14:27:13 | INFO     | Poll 1: still running (0s elapsed). Next check in 20s.


14:27:33 | INFO     | Poll 2: still running (20s elapsed). Next check in 20s.


14:27:54 | INFO     | Training complete after 40s and 3 polls. Model ID: b135ba5b-da0c-457c-8d23-a2f58505b79f


14:27:54 | INFO     | Uploading data, part 1/1 (33770 bytes)



Resilient training complete.
Model ID:    b135ba5b-da0c-457c-8d23-a2f58505b79f
Holdout AUC: 0.9436


---

## Part 6: The Full Resilient Pipeline

### Composing the Patterns

Now we have all the building blocks. The cell below assembles them into a single `run_pipeline` function that:

1. Submits an async training job
2. Polls with error handling and a deadline
3. Runs resilient prediction on the holdout set
4. Persists the model ID and task ID to a registry file
5. Returns structured results

This is roughly what a production batch pipeline would look like -- abstracting the NEXUS API surface behind a function that only surfaces failures worth acting on.

In [15]:
def run_pipeline(
    X_train,
    y_train,
    X_eval,
    y_eval,
    mode="quality",
    registry_path="module_06_registry.json",
    poll_interval=20,
    deadline_seconds=900,
):
    """
    Full resilient pipeline: train, poll, predict, persist.

    Returns a dict with model_id, task_id, auc, and runtime_seconds.
    """
    pipeline_start = time.time()
    log.info(f"Pipeline starting. mode={mode}, features={X_train.shape[1]}")

    # --- Submit -----------------------------------------------------------
    nexus = NEXUSClassifier(mode=mode)
    try:
        task_id = nexus.submit_fit_task(X_train, y_train)
        log.info(f"Task submitted: {task_id}")

    except (TypeError, ValueError) as exc:
        log.error(f"Client-side validation error on submit: {exc}")
        raise

    except RETRYABLE_EXCEPTIONS as exc:
        trace_id = getattr(exc, "trace_id", None)
        log.warning(f"Transient {type(exc).__name__} on submit. trace_id={trace_id}: {exc}")
        raise  # Caller wraps with @with_retry if needed

    except NEXUSError as exc:
        trace_id = getattr(exc, "trace_id", None)
        log.error(f"{type(exc).__name__} on submit. trace_id={trace_id}: {exc}")
        raise

    # --- Poll -------------------------------------------------------------
    resilient_poll_fit(
        nexus, task_id,
        poll_interval=poll_interval,
        deadline_seconds=deadline_seconds,
    )

    # --- Predict ----------------------------------------------------------
    @with_retry(max_retries=3, base_delay=5.0)
    def predict():
        return nexus.predict_proba(X_eval)

    proba = predict()
    auc   = roc_auc_score(y_eval, proba[:, 1])

    # --- Persist ----------------------------------------------------------
    runtime = time.time() - pipeline_start
    result  = {
        "model_id":        nexus.trained_model_id_,
        "task_id":         task_id,
        "auc":             round(auc, 4),
        "mode":            mode,
        "features":        X_train.shape[1],
        "runtime_seconds": round(runtime, 1),
    }

    try:
        with open(registry_path, "r") as f:
            registry = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        registry = []

    registry.append(result)
    with open(registry_path, "w") as f:
        json.dump(registry, f, indent=2)

    log.info(f"Pipeline complete. model_id={result['model_id']} AUC={result['auc']} runtime={runtime:.0f}s")
    return result


print("run_pipeline defined.")

run_pipeline defined.


In [16]:
# Execute the full pipeline
result = run_pipeline(
    X_train, y_train,
    X_holdout, y_holdout,
    mode="quality",
    registry_path="module_06_registry.json",
)

print("\nPipeline results:")
for k, v in result.items():
    print(f"  {k:<25}: {v}")

14:28:07 | INFO     | Pipeline starting. mode=quality, features=15


14:28:08 | INFO     | Uploading data, part 1/1 (93912 bytes)


14:28:08 | INFO     | Uploading data, part 1/1 (1887 bytes)


14:28:08 | INFO     | Task submitted: 750f4f3a138609b1d8db7e3c4a532e3a


14:28:08 | INFO     | Starting resilient poll for task 750f4f3a138609b1d8db7e3c4a532e3a


...

14:28:29 | INFO     | Poll 2: still running (20s elapsed). Next check in 20s.


14:28:49 | INFO     | Training complete after 40s and 3 polls. Model ID: 8a10b687-bc0b-498b-bf4a-eebb06209607


14:28:49 | INFO     | Uploading data, part 1/1 (33770 bytes)


14:29:01 | INFO     | Pipeline complete. model_id=8a10b687-bc0b-498b-bf4a-eebb06209607 AUC=0.9442 runtime=54s



Pipeline results:
  model_id                 : 8a10b687-bc0b-498b-bf4a-eebb06209607
  task_id                  : 750f4f3a138609b1d8db7e3c4a532e3a
  auc                      : 0.9442
  mode                     : quality
  features                 : 15
  runtime_seconds          : 53.5


---

## Summary: The Resilience Toolkit

Here is a complete reference of the patterns covered in this module:

| Pattern | When to use | Key detail |
|---------|-------------|------------|
| **Type-based catch** | Always | Catch the typed exceptions; `isinstance(exc, RETRYABLE_EXCEPTIONS)` decides retry vs. fix vs. escalate |
| **Exponential backoff** | Transient errors | Double the wait each retry; add jitter |
| **`@with_retry` decorator** | Repeated NEXUS calls | Wraps any function; retries only on the four retryable exception types |
| **Deadline-based polling** | Long-running operations | Set a wall-clock deadline; the server manages its own timeouts |
| **Resilient poll loop** | Async workflows | Deadline + consecutive error cap; always preserve task ID |
| **Registry persistence** | Production pipelines | Store task ID + model ID before polling completes |

The single most important rule: **never discard a task ID because polling failed**. The training job is running on Fundamental's infrastructure and is recoverable. The task ID is your lifeline back to that work.

---

## Save Your Model ID

Save the model trained in this module -- you will use it in Module 7 when we cover diagnostics and debugging.

In [17]:
# Store the resilient model ID for Modules 7 and 8
MODULE6_MODEL_ID = nexus_resilient.trained_model_id_
save_state("MODULE6_MODEL_ID", MODULE6_MODEL_ID)
save_state("TOP_FEATURES", TOP_FEATURES)

print("=== Save These Values ===")
print(f"Resilient model (quality): {nexus_resilient.trained_model_id_}")
print(f"Pipeline model (quality):  {result['model_id']}")
print()
print(f"Registry file: module_06_registry.json")
print(f"Registry contents:")
with open("module_06_registry.json") as f:
    print(json.dumps(json.load(f), indent=2))

Saved 'MODULE6_MODEL_ID' to workshop state.
Saved 'TOP_FEATURES' to workshop state.
=== Save These Values ===
Resilient model (quality): b135ba5b-da0c-457c-8d23-a2f58505b79f
Pipeline model (quality):  8a10b687-bc0b-498b-bf4a-eebb06209607

Registry file: module_06_registry.json
Registry contents:
[
  {
    "model_id": "06023690-b40c-4639-8281-cc42732152ee",
    "task_id": "73f1fca770cba2978f98b837fa079b9b",
    "auc": 0.9406,
    "mode": "quality",
    "features": 15,
    "runtime_seconds": 55.9
  },
  {
    "model_id": "8a10b687-bc0b-498b-bf4a-eebb06209607",
    "task_id": "750f4f3a138609b1d8db7e3c4a532e3a",
    "auc": 0.9442,
    "mode": "quality",
    "features": 15,
    "runtime_seconds": 53.5
  }
]


---

## Key Takeaways

**The error model is type-driven.** Catch on the exception class — `RateLimitError`, `NetworkError`, `RequestTimeoutError`, `ServerError` are retryable; `ValidationError`, `AuthenticationError`, `AuthorizationError`, `NotFoundError` are not. Avoid scraping the message text; use `isinstance`.

**`exc.trace_id` is the support team's anchor.** Always log it when you catch a `NEXUSError`. It's the single most useful field for getting help on a specific failure.

**Exponential backoff with jitter prevents thundering herds.** Doubling the wait time between retries gives the API breathing room. Jitter prevents synchronized retry storms when multiple clients fail at the same moment.

**Use deadline-based polling for long-running operations.** The server manages its own timeouts. Your job is to decide how long you are willing to wait and poll within that budget.

**Never discard a task ID.** Poll failures do not kill the job. Save the task ID before you poll, not after.

---

**What's Next — Module 7: Diagnostics and Debugging**

Module 7 covers what to do when errors are not easily reproduced: activating the SDK's diagnostics mode, reading detailed log files, interpreting the most common failure patterns, and using the `diagnose()` context manager to capture enhanced reports around specific calls.